# 8. Arrhenius kinetics: measuring an activation energy

**Domain:** physical chemistry / reaction engineering.

Rate constants follow

$$k = A\,e^{-E_a/RT} \qquad \Longrightarrow \qquad \ln k = \ln A - \frac{E_a}{R}\cdot\frac{1}{T}$$

so $\ln k$ is linear in $1/T$. The chemist's Arrhenius plot is itself a feature
construction: the useful variable is the **reciprocal** of temperature.

**Goal:** recover $1/T$ unaided, then measure $E_a$ and $A$.

### The two-stage rule

`beamfeat` fits **ridge** regression (`alpha=1.0`) on *standardised* features. For a
single feature that shrinks the coefficient by about $n/(n+\alpha)$ — 0.2% at
$n=600$, but **7% at $n=13$**. Fine for prediction, fatal for measuring a constant.

So every notebook here runs two stages:

1. **`beamfeat` finds the form** — which combination of variables matters.
2. **OLS on that form estimates the constant** — unbiased.

Never read a physical constant off `.equation()`.


## Setup


In [ ]:
%pip install -q "beamfeat[units]" pandas matplotlib seaborn scikit-learn scipy


In [ ]:
import warnings
from scipy import constants as C
from sklearn.linear_model import LinearRegression
from beamfeat import BeamFeatRegressor

warnings.filterwarnings("ignore", message=".*valid feature names.*")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CSV_DIR = Path("csv")
CSV_DIR.mkdir(exist_ok=True)          # created on first run, reused after


def save_csv(frame, name):
    # Write to csv/<name> once, then read back so every run starts from disk.
    path = CSV_DIR / name
    if not path.exists():
        frame.to_csv(path, index=False)
        print(f"wrote  {path}  ({len(frame)} rows)")
    else:
        print(f"cached {path}")
    return pd.read_csv(path)


def raw_coef(m):
    # beamfeat standardises internally; rescale to original units.
    return m.coef_ / m.scaler_.scale_


def pct_err(est, true):
    return 100.0 * (est - true) / true

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(1)


## Simulated kinetics runs

Parameters resemble gas-phase N₂O₅ decomposition: $E_a \approx 103$ kJ/mol,
$A \approx 4.3\times10^{13}$ s⁻¹. 2% multiplicative noise, typical of rate-constant
work.


In [ ]:
Ea_true = 103_000.0          # J/mol
A_true = 4.3e13              # 1/s
N = 400

T = rng.uniform(273, 450, N)
k = A_true * np.exp(-Ea_true / (C.R * T)) * (1 + rng.normal(0, 0.02, N))

kin = save_csv(pd.DataFrame({"T_K": T, "k": k}), "arrhenius.csv")
T = kin.T_K.values
k = kin.k.values

print(f"T range : {T.min():.0f}-{T.max():.0f} K")
print(f"k spans : {k.min():.2e} to {k.max():.2e} 1/s  ({k.max()/k.min():,.0f}x)")
kin.head()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))

ax[0].scatter(T, k, s=6, alpha=.5)
ax[0].set_xlabel("T (K)"); ax[0].set_ylabel("k (1/s)")
ax[0].set_title("Raw — spans 8 orders of magnitude")

ax[1].scatter(T, np.log(k), s=6, alpha=.5)
ax[1].set_xlabel("T (K)"); ax[1].set_ylabel("ln k")
ax[1].set_title("Log target — still curved")

ax[2].scatter(1 / T, np.log(k), s=6, alpha=.5)
ax[2].set_xlabel("1/T (1/K)"); ax[2].set_ylabel("ln k")
ax[2].set_title("Arrhenius coordinates — straight")

plt.tight_layout()
plt.show()


Three panels, one message: the right coordinates turn a curve into a line. Panel two
is why we target $\ln k$; panel three is the feature the search has to find.


## Stage 1 — find the governing variable


In [ ]:
X = T.reshape(-1, 1)
y = np.log(k)

model = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=0).fit(X, y)

print("discovered :", model.formulas())
print("equation   :", model.equation()[:70])
print(f"R^2        : {model.score(X, y):.6f}")


`1/(x0)` — the reciprocal of temperature. The search arrived at the Arrhenius plot's
x-axis on its own.


## Stage 2 — measure Eₐ and A

The intercept is physically meaningful here ($\ln A$), so we keep it. Whether an
intercept belongs is a physics question, not a statistics one.


In [ ]:
ols = LinearRegression().fit((1.0 / T).reshape(-1, 1), y)
Ea_est = -ols.coef_[0] * C.R
A_est = np.exp(ols.intercept_)

print(f"{'':<22}{'recovered':>14}{'true':>14}{'error':>10}")
print("-" * 62)
print(f"{'Ea (J/mol)':<22}{Ea_est:>14,.0f}{Ea_true:>14,.0f}{pct_err(Ea_est, Ea_true):>9.3f}%")
print(f"{'A (1/s)':<22}{A_est:>14.3e}{A_true:>14.3e}{pct_err(A_est, A_true):>9.2f}%")


$E_a$ lands within a fraction of a percent. $A$ is worse — expected, because it is an
extrapolation to $1/T \to 0$, i.e. infinite temperature, far outside the measured
range. **Extrapolated intercepts always carry more uncertainty than slopes.**


## Caution 1: narrow temperature ranges

Arrhenius fits degrade badly when $1/T$ barely varies.


In [ ]:
rows = []
for lo, hi in [(273, 450), (300, 400), (330, 370), (340, 360), (345, 355)]:
    mask = (T >= lo) & (T <= hi)
    o = LinearRegression().fit((1.0 / T[mask]).reshape(-1, 1), y[mask])
    rows.append({
        "window_K": f"{lo}-{hi}",
        "n": int(mask.sum()),
        "span_1/T": f"{(1/T[mask]).max() - (1/T[mask]).min():.2e}",
        "R2": round(o.score((1/T[mask]).reshape(-1, 1), y[mask]), 4),
        "Ea_error": f"{pct_err(-o.coef_[0]*C.R, Ea_true):+.2f}%",
    })

pd.DataFrame(rows)


Every one of those fits has R² above 0.99 inside its own window, and the error grows
tenfold as the window narrows. **A high R² is not evidence that a fitted constant is
trustworthy.** What matters is the range of the independent variable you sampled.


## Caution 2: how much noise before the form is lost?

The narrow-range failure was about design. This one is about measurement quality.


In [ ]:
rows = []
for noise in [0.02, 0.05, 0.10, 0.25, 0.50, 1.00]:
    r = np.random.default_rng(7)
    k_n = A_true * np.exp(-Ea_true / (C.R * T)) * (1 + r.normal(0, noise, len(T)))
    k_n = np.clip(k_n, 1e-30, None)
    y_n = np.log(k_n)

    m = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=0).fit(X, y_n)
    o = LinearRegression().fit((1.0 / T).reshape(-1, 1), y_n)

    rows.append({
        "noise": f"{noise:.0%}",
        "found_1/T": "1/(x0)" in m.formulas(),
        "n_features": len(m.formulas()),
        "Ea_error": f"{pct_err(-o.coef_[0]*C.R, Ea_true):+.2f}%",
    })

pd.DataFrame(rows)


The functional form survives noise levels far beyond anything a real rate measurement
would show. The *constant* degrades first, and gracefully. That ordering is worth
knowing: when data quality drops you lose precision on $E_a$ long before you lose
confidence that the Arrhenius form is right.


## Takeaways

1. Given only $T$, the search recovered $1/T$ — the Arrhenius coordinate — unaided.
2. OLS on that feature measured $E_a$ to well under 1%.
3. **Keep or drop the intercept based on physics.** Here $\ln A$ is real.
4. Slopes are robust; extrapolated intercepts like $A$ are not.
5. A narrow temperature range destroys the estimate while R² stays high. Sampling
   range matters more than fit quality.
